# Notebook 02: Classification Heads on Frozen Encoder

Goal of this notebook: Evaluate different classification head architectures on top of the Moirai foundation model. 

We evaluate four distinct architectures:
1. Mean Pooling
2. Single-Scale Attention
3. Single-Scale Multi-Head Attention (MHA)
4. Hierarchical Multi-Head Attention

In [2]:
import torch
import pandas as pd
from IPython.display import display

from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder
from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from utils import get_lsst_dataloaders, get_z_loaders, grid_search_heads

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]

# Hyperparameters
MOIRAI_BATCH_SIZE = 64
HEAD_BATCH_SIZE = 256 # High batch size since we only train lightweight heads
LR_GRID = [1e-3, 1e-4]
WD_GRID = [0.01, 0.05]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

# Results storage
df_mean = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES_TO_TEST)
df_attn = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_mha = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hier = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_tr, raw_va, raw_te, num_classes = get_lsst_dataloaders(batch_size=MOIRAI_BATCH_SIZE, device=DEVICE)

## The Fast Evaluation Loop
Since Moirai is frozen, we loop over the patch sizes, instantiate Moirai, and **pre-extract the embeddings ($Z$)**. We then pass these $Z$ loaders to the different head architectures.

In [6]:
# We define a function to run the evaluation for a specific patch size
def evaluate_heads_for_patch(patch):
    
    # 1. Instantiate the Frozen Encoder
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch, context_length=36, patch_size=patch, 
        num_samples=100, target_dim=NUM_VARS, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    )
    
    # 2. Pre-Extract Embeddings (Z)
    tr_z, va_z, te_z = get_z_loaders(
        encoder, raw_tr, raw_va, raw_te, 
        head_batch_size=HEAD_BATCH_SIZE, device=DEVICE, remove_last_patch=True, num_vars=NUM_VARS
    )
    
    return tr_z, va_z, te_z

## 1. Mean Pooling Architecture


How it works: The simplest baseline. It takes the contextual patches generated by Moirai and averages them over the temporal dimension. The result is a single flattened vector containing the average representation of each variable, which is then passed to a Linear classifier.

In [7]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch) # Extract once
    
    _, acc = grid_search_heads(
        MeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes},
        tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
    )
    df_mean.loc["Mean Pooling", patch] = acc

display(df_mean.astype(float).round(4))

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Process 34499 has 862.00 MiB memory in use. Process 38140 has 850.00 MiB memory in use. Including non-PyTorch memory, this process has 12.89 GiB memory in use. Of the allocated memory 11.80 GiB is allocated by PyTorch, and 991.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 2. Single-Scale Attention Architecture


How it works: Instead of a naive average, this head uses a learned Query vector. This Query looks at all the patches (Keys/Values) and assigns an attention weight to each. The network learns which patches are the most important for the classification task.
* shared_context: One single Query acts across all variables simultaneously.
* independent_context: Each variable has its own dedicated Query to find its own important patches.

In [ ]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            SingleScaleAttentionClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_attn.loc[mode, patch] = acc

display(df_attn.astype(float).round(4))

## 3. Multi-Head Attention (MHA)


How it works: A single attention mechanism might focus entirely on one feature (e.g., the highest peak). Multi-Head Attention solves this by projecting the data into multiple sub-spaces (e.g., 16 heads).
This allows the model to capture multiple distinct temporal concepts simultaneously (e.g., Head 1 looks at recent patches, Head 2 looks at periodic drops, ...).

In [ ]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_mha.loc[mode, patch] = acc

display(df_mha.astype(float).round(4))

## 4. Hierarchical Multi-Head Attention


How it works: Inspired by Hierarchical Attention Networks (HAN). It performs attention in two steps:
1. Temporal Attention: MHA is applied *within* each variable individually to summarize its temporal patches into a single variable-vector.
2. Variable Attention: A second MHA is applied *across* the summarized variables. This allows the network to learn inter-variable correlations.

In [ ]:
for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    
    for mode in MODES:
        _, acc = grid_search_heads(
            HierarchicalMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        df_hier.loc[mode, patch] = acc

display(df_hier.astype(float).round(4))

## Final Summary
Review of all architectures evaluated on the frozen Moirai encoder.

In [ ]:
print("\n RESULTS: MEAN POOLING")
display(df_mean.astype(float).round(4))

print("\n RESULTS: ATTENTION (Base)")
display(df_attn.astype(float).round(4))

print("\n RESULTS: MULTI-HEAD ATTENTION")
display(df_mha.astype(float).round(4))

print("\n RESULTS: HIERARCHICAL MHA")
display(df_hier.astype(float).round(4))